# Model Fitting Demo

Fit TG-SSP, IBP, and NB-SSP on a single UCI experiment, print parameters and predictions, and plot predicted vs. true trajectory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, REPO_ROOT)

from models.tg_ssp import GD
from models.ibp import IBP
from models.nb_ssp import NegBintSBSP

## Load one UCI experiment

In [ ]:
metadata = np.load(os.path.join(REPO_ROOT, 'data/preprocessed/uci/experiments_metadata.npy'),
                   allow_pickle=True)
exp = metadata[6]

D0 = exp['D0']
D1 = exp['D1']
N_pilot = exp['N_pilot']
U_true = exp['U_true']
cum = np.array(exp['cumulative_users'])
counts_short = cum[:D0 + 1]

print(f'Experiment 6: D0={D0}, D1={D1}, N_pilot={N_pilot:,}, U_true={U_true:,}')

## Fit TG-SSP

In [ ]:
gd = GD()
gd_params = gd.regression(counts_short, num_its=5, norm=2, status=False)
gd_pred = gd.mean(D0, D1, N_pilot, gd_params)

print(f'TG-SSP parameters: {gd_params}')
print(f'TG-SSP predicted new users at day {D0+D1}: {gd_pred[-1]:.0f}')

## Fit IBP

In [ ]:
ibp = IBP()
ibp_params = ibp.regression(counts_short, num_its=5, norm=2, status=False)
ibp_pred = ibp.mean(D0, D1, ibp_params)

print(f'IBP parameters: {ibp_params}')
print(f'IBP predicted new users at day {D0+D1}: {ibp_pred[-1]:.0f}')

## Fit NB-SSP

In [ ]:
nbp = NegBintSBSP()
nbp_params = nbp.fit_regression(D0, N_pilot, counts_short, num_its=5)
nbp_pred = nbp.mean_number_new_users(D0, D1, N_pilot, nbp_params)

print(f'NB-SSP parameters: {nbp_params}')
print(f'NB-SSP predicted new users at day {D0+D1}: {nbp_pred[-1]:.0f}')

## Plot: Predicted vs. True Trajectory

In [ ]:
from plotting.style import COLORS, LABELS, apply_style
apply_style()

fig, ax = plt.subplots(figsize=(8, 5))

# Truth
days = np.arange(len(cum))
ax.plot(days, cum, 'k-', linewidth=2, label='Truth')
ax.axvline(x=D0, color='gray', linestyle=':', alpha=0.7)
ax.axvspan(0, D0, alpha=0.08, color='gray')

# Predictions (new users added to N_pilot)
pred_days = np.arange(D0 + 1, D0 + 1 + D1)
ax.plot(pred_days, N_pilot + gd_pred, '--', color=COLORS['TG_SSP'], label=LABELS['TG_SSP'])
ax.plot(pred_days, N_pilot + ibp_pred, '-.', color=COLORS['IBP'], label=LABELS['IBP'])
ax.plot(pred_days, N_pilot + nbp_pred, '-', color=COLORS['NB_SSP_regression'], label=LABELS['NB_SSP_regression'])

ax.set_xlabel('Day')
ax.set_ylabel('Cumulative distinct users')
ax.set_title(f'UCI Experiment 6: Predicted vs. True')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()